# 🗂️ Thử nghiệm Hierarchical Chunking cho Luật Giao thông
**Mục tiêu:** Kiểm tra trực quan kết quả chia tài liệu pháp luật theo cấu trúc phân cấp (Chương → Điều → Khoản → Điểm). Giúp xác nhận module `HierarchicalLegalSplitter` đang hoạt động đúng trước khi đưa vào pipeline chính.

**Tài liệu sử dụng:** `Data/cleaned/*.txt` (Luật TTATGT 2024, NĐ 100/2019, NĐ 123/2021)

In [ ]:
import os
import sys
import json
from collections import Counter

# Thêm project root vào sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f" Project root: {PROJECT_ROOT}")

from source.chunking.hierarchical_splitter import HierarchicalLegalSplitter
print(" Import HierarchicalLegalSplitter thành công!")

✅ Project root: /media/pphong/D:/Do_An_Tot_Nghiep/GitHub1


ModuleNotFoundError: No module named 'source'

## 1. Load và Chia tài liệu

In [ ]:
DATA_DIR = os.path.join(PROJECT_ROOT, "Data", "cleaned")

# Ánh xạ file → tên văn bản hiển thị
DOCS = {
    "nd100_2019.txt": "Nghị định 100/2019/NĐ-CP",
    "nd123_2021.txt": "Nghị định 123/2021/NĐ-CP",
    "luat_ttatgt_2024.txt": "Luật Trật tự An toàn Giao thông đường bộ 2024",
}

all_chunks = {}
for filename, doc_name in DOCS.items():
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f"⚠️  Không tìm thấy: {filepath}")
        continue
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    
    splitter = HierarchicalLegalSplitter(document_name=doc_name)
    chunks = splitter.split_text(text)
    all_chunks[doc_name] = chunks
    print(f"📄 {doc_name}: {len(chunks):,} chunks")

## 2. Kiểm tra cấu trúc chunk và phân phối loại

In [ ]:
for doc_name, chunks in all_chunks.items():
    print(f"\n{'='*60}")
    print(f"📑 VĂN BẢN: {doc_name}")
    print(f"{'='*60}")
    
    # Phân phối loại chunk
    type_counts = Counter(c['metadata']['type'] for c in chunks)
    print(f"\n📊 Phân phối loại chunk:")
    for chunk_type, count in type_counts.most_common():
        bar = '█' * (count // max(1, max(type_counts.values()) // 20))
        print(f"  {chunk_type:10s}: {count:4d}  {bar}")
    
    # Thống kê Chương
    unique_chuong = set(c['metadata']['chuong'] for c in chunks if c['metadata']['chuong'] != 'N/A')
    print(f"\n🏛️  Số Chương phát hiện được: {len(unique_chuong)}")
    
    # Thống kê Điều
    unique_dieu = set(c['metadata']['dieu'] for c in chunks if c['metadata']['dieu'] != 'N/A')
    print(f"📋 Số Điều phát hiện được   : {len(unique_dieu)}")

## 3. Xem 5 chunk mẫu từ NĐ 100

In [ ]:
target_doc = "Nghị định 100/2019/NĐ-CP"
if target_doc in all_chunks:
    sample_chunks = all_chunks[target_doc][:5]
    print(f"🔍 5 chunks đầu tiên của {target_doc}:\n")
    for i, chunk in enumerate(sample_chunks, 1):
        meta = chunk['metadata']
        print(f"--- Chunk #{i} ---")
        print(f"  Loại    : {meta['type']}")
        print(f"  Chương  : {meta['chuong']}")
        print(f"  Điều    : {meta['dieu']}")
        print(f"  Khoản   : {meta['khoan']}")
        print(f"  Điểm    : {meta['diem']}")
        print(f"  Nội dung: {chunk['content'][:200]}..." if len(chunk['content']) > 200 else f"  Nội dung: {chunk['content']}")
        print()

## 4. Thử nghiệm tìm kiếm chunk theo Điều cụ thể

In [ ]:
def find_chunks_by_dieu(doc_name: str, dieu_keyword: str):
    """Tìm tất cả chunks thuộc một Điều cụ thể."""
    if doc_name not in all_chunks:
        return []
    return [
        c for c in all_chunks[doc_name]
        if dieu_keyword.lower() in c['metadata']['dieu'].lower()
    ]

# Tìm Điều 6 trong NĐ 100 (quy định xử phạt nồng độ cồn)
results = find_chunks_by_dieu("Nghị định 100/2019/NĐ-CP", "Điều 6")
print(f"🔎 Tìm thấy {len(results)} chunks thuộc 'Điều 6' trong NĐ 100:\n")
for r in results:
    print(f"  [Khoản {r['metadata']['khoan']}, Điểm {r['metadata']['diem']}] {r['content'][:150]}")

## 5. Thống kê độ dài chunk (phát hiện chunk quá ngắn/dài)

In [ ]:
for doc_name, chunks in all_chunks.items():
    lengths = [len(c['content']) for c in chunks]
    avg = sum(lengths) / len(lengths) if lengths else 0
    too_short = sum(1 for l in lengths if l < 30)
    too_long = sum(1 for l in lengths if l > 1000)
    
    print(f"📄 {doc_name}")
    print(f"   Min: {min(lengths):5d} ký tự | Max: {max(lengths):5d} ký tự | Avg: {avg:.0f} ký tự")
    print(f"   ⚠️  Quá ngắn (<30 ký tự): {too_short} chunks")
    print(f"   ⚠️  Quá dài  (>1000 ký tự): {too_long} chunks")
    print()

print("✅ Kiểm nghiệm Hierarchical Chunking hoàn tất!")